In [1]:
from __future__ import annotations

import json
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import pandas as pd

try:
    import duckdb
except Exception as exc:
    raise RuntimeError("Install duckdb in this notebook environment to run this database notebook.") from exc

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

@dataclass
class DuckDBConfig:
    chunks_path: Path = ROOT / "data" / "processed_reports" / "chunks" / "text_chunks.parquet"
    manifest_path: Path = ROOT / "data" / "processed_reports" / "page_or_sheet_manifest.parquet"
    structured_path: Path = ROOT / "data" / "processed_reports" / "tables" / "structured_long.parquet"
    metric_dictionary_path: Path = ROOT / "data" / "processed_reports" / "tables" / "metric_dictionary.parquet"
    semantic_metrics_path: Path = ROOT / "data" / "processed_reports" / "tables" / "semantic_metrics.parquet"
    vector_records_path: Path = ROOT / "data" / "processed_reports" / "semantic_vector_records.parquet"
    duckdb_path: Path = ROOT / "storage" / "insightviewer_semantic.duckdb"

CFG = DuckDBConfig()
CFG.duckdb_path.parent.mkdir(parents=True, exist_ok=True)
print(f"DuckDB target: {CFG.duckdb_path}")

DuckDB target: /home/ssever/InsightViewer/storage/insightviewer_semantic.duckdb


## **Load Semantic Artifacts**

In [2]:
def read_dataframe(path: Path) -> pd.DataFrame:
    if path.exists():
        if path.suffix == ".parquet":
            return pd.read_parquet(path)
        if path.suffix == ".csv":
            return pd.read_csv(path)

    csv_fallback = path.with_suffix(".csv")
    if csv_fallback.exists():
        return pd.read_csv(csv_fallback)

    print(f"Missing artifact: {path}")
    return pd.DataFrame()


tables = {
    "metric_dictionary": read_dataframe(CFG.metric_dictionary_path),
    "semantic_metrics": read_dataframe(CFG.semantic_metrics_path),
    "semantic_vector_records": read_dataframe(CFG.vector_records_path),
    "page_or_sheet_manifest": read_dataframe(CFG.manifest_path),
    "text_chunks": read_dataframe(CFG.chunks_path),
    "structured_long": read_dataframe(CFG.structured_path),
}

for table_name, frame in tables.items():
    print(table_name, frame.shape)

metric_dictionary (41, 6)
semantic_metrics (91, 30)
semantic_vector_records (490, 4)
page_or_sheet_manifest (70, 24)
text_chunks (399, 22)
structured_long (91, 21)


## **Materialize Tables and Views**

In [3]:
def normalize_for_duckdb(frame: pd.DataFrame) -> pd.DataFrame:
    """Convert pandas dtypes/values that older DuckDB builds cannot register directly."""
    normalized = frame.copy()
    for column in normalized.columns:
        series = normalized[column]
        dtype_name = str(series.dtype)

        if dtype_name in {"str", "string"} or dtype_name.startswith("string["):
            normalized[column] = series.astype("object").where(series.notna(), None)
            continue

        if series.dtype == "object":
            normalized[column] = series.map(coerce_duckdb_object)

    return normalized


def coerce_duckdb_object(value: Any) -> Any:
    if value is None:
        return None
    if isinstance(value, (dict, list, tuple, set)):
        return json.dumps(value, default=str)
    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass
    return value


def write_table(conn: duckdb.DuckDBPyConnection, table_name: str, frame: pd.DataFrame) -> None:
    frame_for_duckdb = normalize_for_duckdb(frame)
    conn.register("_frame", frame_for_duckdb)
    try:
        conn.execute(f"CREATE OR REPLACE TABLE {table_name} AS SELECT * FROM _frame")
    finally:
        conn.unregister("_frame")


conn = duckdb.connect(str(CFG.duckdb_path))
for table_name, frame in tables.items():
    write_table(conn, table_name, frame)

conn.execute("""
    CREATE OR REPLACE VIEW metric_timeseries AS
    SELECT
        entity,
        ticker,
        metric_canonical,
        metric_display_name,
        fiscal_year_semantic AS fiscal_year,
        fiscal_quarter_semantic AS fiscal_quarter,
        period_label,
        period_sort,
        units,
        currency,
        SUM(value) AS value,
        COUNT(*) AS source_row_count
    FROM semantic_metrics
    WHERE value IS NOT NULL
    GROUP BY
        entity, ticker, metric_canonical, metric_display_name, fiscal_year_semantic,
        fiscal_quarter_semantic, period_label, period_sort, units, currency
""")

conn.execute("""
    CREATE OR REPLACE VIEW retrieval_catalog AS
    SELECT
        vector_id,
        content_kind,
        document,
        metadata_json
    FROM semantic_vector_records
""")

print(f"Wrote DuckDB semantic database: {CFG.duckdb_path}")

Wrote DuckDB semantic database: /home/ssever/InsightViewer/storage/insightviewer_semantic.duckdb


## **Query Examples**

In [4]:
# Example query function to get a metric timeseries for an entity, optionally filtered by ticker
def available_metric_filters() -> pd.DataFrame:
    return conn.execute("""
        SELECT
            metric_canonical,
            ticker,
            entity,
            COUNT(*) AS row_count
        FROM metric_timeseries
        GROUP BY metric_canonical, ticker, entity
        ORDER BY metric_canonical, ticker, entity
    """).df()


def query_metric_timeseries(metric: str, ticker: str | None = None) -> pd.DataFrame:
    sql = """
        SELECT *
        FROM metric_timeseries
        WHERE metric_canonical = ?
    """
    params: list[Any] = [metric]
    if ticker:
        sql += " AND ticker = ?"
        params.append(ticker)
    sql += " ORDER BY period_sort"
    return conn.execute(sql, params).df()


display(available_metric_filters())
query_metric_timeseries("revenue", ticker="MSFT").head(20)

,metric_canonical,ticker,entity,row_count
0,amount,MSFT,Microsoft,3
1,dividend_period,MSFT,Microsoft,1


,entity,ticker,metric_canonical,metric_display_name,fiscal_year,fiscal_quarter,period_label,period_sort,units,currency,value,source_row_count


In [5]:
conn.execute("""
    SELECT *
    FROM metric_timeseries
    WHERE metric_canonical = 'revenue'
    LIMIT 20
""").df()


,entity,ticker,metric_canonical,metric_display_name,fiscal_year,fiscal_quarter,period_label,period_sort,units,currency,value,source_row_count


In [6]:
conn.close()